In [7]:
!pip install tensorflow


In [24]:
import zipfile
import os

with zipfile.ZipFile("turkishsentimentanalysisdataset.zip", 'r') as zip_ref:
    zip_ref.extractall("dataset")

# Klasördeki dosya isimlerine bakalım
for root, dirs, files in os.walk("dataset"):
    for file in files:
        print(file)


test.csv
train.csv
Twitter Sentiment Analysis.ipynb
README.md
TurkishTweets.csv
tweetset.csv


In [25]:
# Kaggle API bağlantısı
from google.colab import files
files.upload()  # kaggle.json dosyasını buradan yükle

!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

# Veri setini indir
!kaggle datasets download -d winvoker/turkishsentimentanalysisdataset

# ZIP açma
import zipfile
with zipfile.ZipFile("turkish-tweets-sentiment-analysis-main.zip", "r") as zip_ref:
    zip_ref.extractall("dataset")

# Dosya içeriğini göster
import pandas as pd

df = pd.read_csv("dataset/train.csv")
df.head()


Saving kaggle.json to kaggle (4).json
Dataset URL: https://www.kaggle.com/datasets/winvoker/turkishsentimentanalysisdataset
License(s): CC-BY-SA-4.0
turkishsentimentanalysisdataset.zip: Skipping, found more recently modified local copy (use --force to force download)


,text,label,dataset
0,ürünü hepsiburadadan alalı 3 hafta oldu. orjin...,Positive,urun_yorumlari
1,"ürünlerden çok memnunum, kesinlikle herkese ta...",Positive,urun_yorumlari
2,"hızlı kargo, temiz alışveriş.teşekkür ederim.",Positive,urun_yorumlari
3,Çünkü aranan tapınak bu bölgededir .,Notr,wiki
4,bu telefonu başlıca alma nedenlerim ise elimde...,Positive,urun_yorumlari


In [26]:
# Gerekli kütüphaneler
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense

# 🔹 Veri Yükleme
df = pd.read_csv("/content/dataset/train.csv")  # YOLU GÜNCELLE
df = df[['text', 'label']].dropna()

# 🔹 Etiketleri sayısallaştır
df['label'] = df['label'].replace({'Negative': 0, 'Notr': 1, 'Positive': 2})

# 🔹 Eğitim/test böl
X = df['text']
y = df['label']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 🔸 TF-IDF + Logistic Regression
turkish_stopwords = [
    'acaba', 'ama', 'aslında', 'az', 'bazı', 'belki', 'biri', 'birkaç', 'birşey',
    'biz', 'bu', 'çok', 'çünkü', 'da', 'daha', 'de', 'defa', 'diye', 'en', 'gibi',
    'hem', 'hep', 'hepsi', 'her', 'hiç', 'için', 'ile', 'ise', 'kez', 'ki', 'kim',
    'mı', 'mu', 'mü', 'nasıl', 'ne', 'neden', 'nerde', 'nerede', 'nereye', 'niçin',
    'niye', 'o', 'sanki', 'şey', 'siz', 'şu', 'tüm', 've', 'veya', 'ya', 'yani'
]

vectorizer = TfidfVectorizer(max_features=10000, stop_words=turkish_stopwords)
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

model_tfidf = LogisticRegression(max_iter=1000)
model_tfidf.fit(X_train_tfidf, y_train)
y_pred_tfidf = model_tfidf.predict(X_test_tfidf)

print("🔹 TF-IDF + Logistic Regression Sonuçları:\n")
print(classification_report(y_test, y_pred_tfidf, target_names=["Negative", "Notr", "Positive"]))

# 🔸 LSTM + Embedding
max_words = 20000
max_len = 100

tokenizer = Tokenizer(num_words=max_words)
tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)

X_train_pad = pad_sequences(X_train_seq, maxlen=max_len)
X_test_pad = pad_sequences(X_test_seq, maxlen=max_len)

model_lstm = Sequential()
model_lstm.add(Embedding(input_dim=max_words, output_dim=128, input_length=max_len))
model_lstm.add(LSTM(64, dropout=0.2, recurrent_dropout=0.2))
model_lstm.add(Dense(3, activation='softmax'))  # 3 sınıf

model_lstm.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model_lstm.summary()

# Eğitim
model_lstm.fit(X_train_pad, y_train, epochs=3, batch_size=256, validation_split=0.1)

# Değerlendirme
y_pred_probs = model_lstm.predict(X_test_pad)
y_pred_lstm = np.argmax(y_pred_probs, axis=1)

print("\n🔹 LSTM + Embedding Sonuçları:\n")
print(classification_report(y_test, y_pred_lstm, target_names=["Negative", "Notr", "Positive"]))


<ipython-input-26-55bdb27d09da>:18: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['label'] = df['label'].replace({'Negative': 0, 'Notr': 1, 'Positive': 2})


🔹 TF-IDF + Logistic Regression Sonuçları:

              precision    recall  f1-score   support

    Negative       0.86      0.69      0.77     10100
        Notr       0.95      0.98      0.96     30657
    Positive       0.94      0.95      0.94     47379

    accuracy                           0.93     88136
   macro avg       0.91      0.87      0.89     88136
weighted avg       0.93      0.93      0.93     88136



/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

Epoch 1/3
1240/1240 ━━━━━━━━━━━━━━━━━━━━ 395s 316ms/step - accuracy: 0.8799 - loss: 0.3209 - val_accuracy: 0.9410 - val_loss: 0.1585
Epoch 2/3
1240/1240 ━━━━━━━━━━━━━━━━━━━━ 436s 311ms/step - accuracy: 0.9516 - loss: 0.1348 - val_accuracy: 0.9430 - val_loss: 0.1571
Epoch 3/3
1240/1240 ━━━━━━━━━━━━━━━━━━━━ 442s 312ms/step - accuracy: 0.9599 - loss: 0.1103 - val_accuracy: 0.9436 - val_loss: 0.1569
2755/2755 ━━━━━━━━━━━━━━━━━━━━ 147s 53ms/step

🔹 LSTM + Embedding Sonuçları:

              precision    recall  f1-score   support

    Negative       0.86      0.75      0.80     10100
        Notr       0.97      0.98      0.98     30657
    Positive       0.94      0.96      0.95     47379

    accuracy                           0.94     88136
   macro avg       0.92      0.90      0.91     88136
weighted avg       0.94      0.94      0.94     88136



In [27]:
# Tensorflow projector için embedding matrisini ve kelimeleri dışa aktar

embedding_layer = model_lstm.layers[0]
embedding_weights = embedding_layer.get_weights()[0]

# Tokenizer'dan kelimeleri al
reverse_word_index = dict((i, word) for word, i in tokenizer.word_index.items())

# Kelimeleri yaz
with open("meta.tsv", "w", encoding="utf-8") as f_meta:
    for i in range(1, min(max_words, len(reverse_word_index))):
        word = reverse_word_index.get(i, "")
        f_meta.write(word + "\n")

# Vektörleri yaz
with open("vecs.tsv", "w", encoding="utf-8") as f_vecs:
    for i in range(1, min(max_words, len(reverse_word_index))):
        vector = embedding_weights[i]
        f_vecs.write("\t".join([str(x) for x in vector]) + "\n")


In [28]:
# Hatalı durum: meta.tsv bir satır eksik → sonuna boş satır ekle
with open("meta.tsv", "a", encoding="utf-8") as f:
    f.write("UNKNOWN\n")


In [29]:
from google.colab import files
files.download("meta.tsv")
files.download("vecs.tsv")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Küçük veri seti

In [13]:
# Gerekli kütüphaneler
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense

# 🔹 Veri Yükleme (GÜNCELLENDİ)
df = pd.read_csv("/content/turkish-tweets-sentiment-analysis-main/data/TurkishTweets.csv")
df = df[['Tweet', 'Etiket']].dropna()

# 🔹 Etiketleri sayısal hale getir
df['label'] = df['Etiket'].astype('category').cat.codes  # örn: kızgın=0, mutlu=1, üzgün=2...

X = df['Tweet']
y = df['label']
label_map = dict(enumerate(df['Etiket'].astype('category').cat.categories))  # geri çevrim için

# 🔹 Eğitim ve test böl
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 🔸 TF-IDF + Logistic Regression
turkish_stopwords = ['acaba', 'ama', 'aslında', 'az', 'bazı', 'belki', 'biri', 'birkaç', 'birşey',
    'biz', 'bu', 'çok', 'çünkü', 'da', 'daha', 'de', 'defa', 'diye', 'en', 'gibi', 'hem', 'hep', 'hepsi',
    'her', 'hiç', 'için', 'ile', 'ise', 'kez', 'ki', 'kim', 'mı', 'mu', 'mü', 'nasıl', 'ne', 'neden',
    'nerde', 'nerede', 'nereye', 'niçin', 'niye', 'o', 'sanki', 'şey', 'siz', 'şu', 'tüm', 've', 'veya', 'ya', 'yani']

vectorizer = TfidfVectorizer(max_features=5000, stop_words=turkish_stopwords)
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

model_tfidf = LogisticRegression(max_iter=1000)
model_tfidf.fit(X_train_tfidf, y_train)
y_pred_tfidf = model_tfidf.predict(X_test_tfidf)

print("🔹 TF-IDF + Logistic Regression Sonuçları:\n")
print(classification_report(y_test, y_pred_tfidf, target_names=list(label_map.values())))

# 🔸 Tokenizer + Padding
max_words = 10000
max_len = 100

tokenizer = Tokenizer(num_words=max_words)
tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)

X_train_pad = pad_sequences(X_train_seq, maxlen=max_len)
X_test_pad = pad_sequences(X_test_seq, maxlen=max_len)

# 🔸 LSTM + Random Embedding
model_lstm = Sequential()
model_lstm.add(Embedding(input_dim=max_words, output_dim=128, input_length=max_len))
model_lstm.add(LSTM(64, dropout=0.2, recurrent_dropout=0.2))
model_lstm.add(Dense(len(label_map), activation='softmax'))  # çok sınıflı çıktı

model_lstm.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model_lstm.summary()

# 🔸 Eğitim
model_lstm.fit(X_train_pad, y_train, epochs=3, batch_size=128, validation_split=0.1)

# 🔸 Değerlendirme
y_pred_probs = model_lstm.predict(X_test_pad)
y_pred_lstm = np.argmax(y_pred_probs, axis=1)

print("\n🔹 LSTM + Embedding Sonuçları:\n")
print(classification_report(y_test, y_pred_lstm, target_names=list(label_map.values())))



🔹 TF-IDF + Logistic Regression Sonuçları:

              precision    recall  f1-score   support

       korku       0.98      0.98      0.98       164
      kızgın       0.99      0.99      0.99       169
       mutlu       0.97      0.97      0.97       160
     surpriz       0.97      0.97      0.97       156
       üzgün       0.98      0.97      0.98       151

    accuracy                           0.98       800
   macro avg       0.98      0.98      0.98       800
weighted avg       0.98      0.98      0.98       800



/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

Epoch 1/3
23/23 ━━━━━━━━━━━━━━━━━━━━ 16s 363ms/step - accuracy: 0.3133 - loss: 1.5976 - val_accuracy: 0.7500 - val_loss: 1.5330
Epoch 2/3
23/23 ━━━━━━━━━━━━━━━━━━━━ 8s 344ms/step - accuracy: 0.8335 - loss: 1.4458 - val_accuracy: 0.8219 - val_loss: 1.1536
Epoch 3/3
23/23 ━━━━━━━━━━━━━━━━━━━━ 9s 288ms/step - accuracy: 0.8901 - loss: 0.8624 - val_accuracy: 0.9094 - val_loss: 0.5288
25/25 ━━━━━━━━━━━━━━━━━━━━ 2s 52ms/step

🔹 LSTM + Embedding Sonuçları:

              precision    recall  f1-score   support

       korku       0.95      0.96      0.95       164
      kızgın       0.95      0.95      0.95       169
       mutlu       0.95      0.91      0.93       160
     surpriz       0.91      0.94      0.92       156
       üzgün       0.88      0.89      0.88       151

    accuracy                           0.93       800
   macro avg       0.93      0.93      0.93       800
weighted avg       0.93      0.93      0.93       800

